# Barkeep Protocol — Act 1.3 Training Notebook

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from src.dataset_wiki import make_wiki_dataset
from src.model import Transformer
from src.optimizer import AdamW
from src.train import train_model, TrainConfig
from src.analysis import run_all_diagnostics, load_runs, plot_run_comparison

## Config

In [ ]:
import torch
# ── device ──
if torch.xpu.is_available():
    DEVICE = "xpu"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(f"Using device: {DEVICE}")

---
## Hyperparamters

In [ ]:
SEED = 42
generator_global = torch.Generator(device=DEVICE).manual_seed(SEED)

In [ ]:
# ── model hyperparameters ──
CONTEXT_SIZE   = 256
VOCAB_SIZE     = 3000
EMBEDDING_DIM  = 128
NUM_HEADS      = 4
DIM_QK         = 16   # d_k per head
DIM_V          = 32   # d_v per head
FFN_DIM        = 512
NUM_BLOCKS     = 4
DROPOUT        = 0.2
MAX_NORM       = 1

# ── optimizer hyperparameters ──
BETA_1         = 0.9
BETA_2         = 0.99
EPSILON        = 1e-8
DECAY_RATE     = 0.1

# ── training hyperparameters ──
BATCH_SIZE     = 128
STARTING_LR    = 3e-3
ENDING_LR      = 3e-4
WARMUP_STEPS   = 500
LOG_INTERVAL   = 500
NUM_ITRNS      = 5000

In [ ]:
config = TrainConfig(
    log_path     = "runs.jsonl",
    lr_start     = STARTING_LR,
    lr_end       = ENDING_LR,
    iterations   = NUM_ITRNS,
    log_interval = LOG_INTERVAL,
    batch_size   = BATCH_SIZE,
    max_norm     = MAX_NORM,
    warmup_steps = WARMUP_STEPS,
)

In [ ]:
# ── dataset ──
DATA_DIR = r"D:\Bar-Eden\Datasets\Wikitext-2"
data = make_wiki_dataset(context_size=CONTEXT_SIZE, vocab_size=VOCAB_SIZE, data_dir=DATA_DIR, device=DEVICE, seed=SEED)

trn       = data["train"]
dev       = data["dev"]
test      = data["test"]

VOCAB_SIZE = data["vocab_size"]

print(f"Vocab size: {VOCAB_SIZE}")
print("Train:", trn.inputs.shape)
print("Dev:  ", dev.inputs.shape)

In [ ]:
# ── diagnostic batch (shared across all models) ──
DIAG_X = trn.inputs[:512]
DIAG_Y = trn.targets[:512]

---
## Transformer

In [ ]:
transformer = Transformer(
    CONTEXT_SIZE, VOCAB_SIZE, EMBEDDING_DIM,
    dim_qk=DIM_QK,
    dim_v=DIM_V,
    num_heads=NUM_HEADS,
    ffn_dim=FFN_DIM,
    n_blocks=NUM_BLOCKS,
    device=DEVICE, generator=generator_global, p=DROPOUT,
)
print("Optimized Transformer\n", transformer.config_dict())
print(f"Parameters:     {sum(p.nelement() for p in transformer.parameters()):,}")

---
## Optimizer

In [ ]:
optimizer = AdamW(transformer.parameters(), (BETA_1, BETA_2), EPSILON, DECAY_RATE)

---
## Training

In [ ]:
results = train_model(SEED, transformer, optimizer, trn, dev, config, "Wikitext Transformer", DEVICE)
torch.save(results, "Training results/wikitext_transformer_results.pt")
torch.save(transformer, "Model Instances/wikitext_transformer.pt")

In [ ]:
wikitext_transformer_model   = torch.load("Model Instances/wikitext_transformer.pt", weights_only=False)
wikitext_transformer_results = torch.load("Training results/wikitext_transformer_results.pt")

In [ ]:
trn  = torch.load("../../Scratch/Encoded Wikitext/training_set.pt", weights_only=False)
dev  = torch.load("../../Scratch/Encoded Wikitext/dev_set.pt", weights_only=False)
test = torch.load("../../Scratch/Encoded Wikitext/test_set.pt", weights_only=False)

In [ ]:
runs = load_runs("runs.jsonl")
plot_run_comparison(runs, metric="dev_loss")
plot_run_comparison(runs, metric="train_loss")

In [ ]:
run_all_diagnostics(wikitext_transformer_model, wikitext_transformer_results, DIAG_X, DIAG_Y, "Wikitext Transformer")

---
## Sampling/Generating text

In [ ]:
def generate(model, bpe, prompt, max_new_tokens, context_size, device,
             temperature=1.0, top_k=None):
    model.eval()
    encoded = bpe.encode(prompt)
    original_length = len(encoded)

    for _ in range(max_new_tokens):
        window = encoded[-context_size:]
        pad_len = context_size - len(window)
        padded = [0] * pad_len + window
        input_tokens = torch.tensor([padded], dtype=torch.long, device=device)

        with torch.no_grad():
            logits = model(input_tokens)[0, -1, :]   # (vocab,)

        logits = logits / temperature

        if top_k is not None:
            values, indices = torch.topk(logits, top_k)
            probs = torch.softmax(values, dim=-1)
            sampled = torch.multinomial(probs, num_samples=1)
            next_token = indices[sampled].item()
        else:
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()

        encoded.append(next_token)

    return bpe.decode(encoded[original_length:])

In [ ]:
from src.bpe import ByteBPE

bpe = ByteBPE()
bpe.load(r"D:\Bar-Eden\Datasets\Wikitext-2\bpe.json")

In [ ]:
print(generate(wikitext_transformer_model, bpe, "The history of", 400000, 256, "cpu", temperature=0.8, top_k=40))

---